# Decoding strategies (greedy, beam, top-k, nucleus, temperature) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Turn model logits into text by choosing how much probability mass to explore
The five examples compare greedy choice, beam search scoring, temperature, top-k and nucleus filtering on fixed logits.

In [ ]:
logits=np.array([3.,2.,0.5]); p=softmax(logits); tokens=['A','B','C']
plt.figure(figsize=(5,3)); plt.bar(tokens,p); plt.title('Greedy takes the largest probability')
assert tokens[int(np.argmax(p))]=='A'

In [ ]:
seqs=['AA','AB','BA']; probs=np.array([0.6*0.5,0.6*0.4,0.3*0.9]); scores=np.log(probs)
plt.figure(figsize=(5,3)); plt.bar(seqs,scores); plt.title('Beam search keeps high log-probability paths')
assert seqs[int(np.argmax(scores))]=='AA'

In [ ]:
base=np.array([2.,1.,0.]); temps=[0.5,1,2]; ent=[]
for t in temps:
    p=softmax(base/t); ent.append(-(p*np.log(p)).sum())
plt.figure(figsize=(5,3)); plt.plot(temps,ent,'-o'); plt.title('Higher temperature raises entropy')
assert ent[0]<ent[-1]

In [ ]:
p=softmax(np.array([4,3,2,1,0.],float)); k=3; filt=np.r_[p[:k],np.zeros(len(p)-k)]; filt=filt/filt.sum()
plt.figure(figsize=(5,3)); plt.bar(range(5),filt); plt.title('Top-k keeps exactly k candidates')
assert np.count_nonzero(filt)==3 and abs(filt.sum()-1)<1e-9

In [ ]:
p=softmax(np.array([4,3,2,1,0.],float)); order=np.argsort(-p); c=np.cumsum(p[order]); keep=order[c<=0.9].tolist(); keep.append(order[len(keep)]); mask=np.zeros_like(p); mask[keep]=p[keep]; mask/=mask.sum()
plt.figure(figsize=(5,3)); plt.bar(range(5),mask); plt.title('Nucleus keeps the smallest set above p')
assert mask.sum()>0.999 and np.count_nonzero(mask)==3